# OpenAI Agents: actor y handoff · 2.1.4

Una implementación de actor, bajo la misma orquestación LangGraph y el mismo RunResult.

In [2]:
import agentic_systems as toolkit

assert toolkit.__version__ == '2.1.4'

In [3]:
@toolkit.tool
def certify(value: int) -> dict:
    return {"value": value, "verified": True}


capability = toolkit.skill(name="evidence", tools=(certify,))

In [4]:
actor_framework = "openai-agents"
runtime = toolkit.runtime(provider="python-runtime")

In [5]:
actor = toolkit.Agent(
    "verification",
    runtime=runtime,
    framework=toolkit.framework(actor_framework),
    skills=(capability,),
    instructions="Call certify once with the supplied value. The Tool output is the result.",
    contract={
        "tool_expectation": {"all_of": ["certify"]},
        "completion": "when_required_tools_satisfied",
    },
    policy={"repair": False},
)
result = actor.run({"value": 37}, timeout_s=120)
events = [event for node in result.walk() for event in node.tool_events]
assert result.ok and result.check_invariants().ok, result.to_dict()
assert any(
    (e.name == "certify" and e.ok and (e.input == {"value": 37}) for e in events)
)
print(
    {
        "input": {"value": 37},
        "process": "runtime + actor -> Tool event",
        "output": result.data,
    }
)

{'input': {'value': 37}, 'process': 'runtime + actor -> Tool event', 'output': {'value': 37, 'verified': True}}


## Handoff, sesión y guardrail reales

La construcción SDK y toolkit está visible. Necesita OpenAI configurado y campaña autorizada. Esta celda live se omite offline; el guardrail negativo bloquea antes del modelo.

In [ ]:
from dotenv import load_dotenv

load_dotenv()
import os
from uuid import uuid4
from agents import Agent as OpenAIAgent, OpenAIChatCompletionsModel, ModelSettings
from agents import GuardrailFunctionOutput, input_guardrail, handoff, SQLiteSession
from openai import AsyncOpenAI

model_id = os.environ.get("OPENAI_MODEL_ID") or os.environ["OPENAI_MODEL"]
endpoint = os.environ.get("OPENAI_BASE_URL") or "https://api.openai.com/v1"
client = AsyncOpenAI(
    api_key=os.environ["OPENAI_API_KEY"], base_url=endpoint, max_retries=0, timeout=30
)
specialist = OpenAIAgent(
    name="billing_specialist",
    model=OpenAIChatCompletionsModel(model=model_id, openai_client=client),
    instructions="You own the response after handoff. Reply exactly SPECIALIST_OK.",
    model_settings=ModelSettings(max_tokens=128),
)
observed = []


def record_handoff(context):
    observed.append({"from": "triage", "to": "billing_specialist"})


transfer = handoff(specialist, on_handoff=record_handoff)


@input_guardrail(run_in_parallel=False)
def block(context, agent, request):
    return GuardrailFunctionOutput(
        output_info={"checked": True},
        tripwire_triggered="BLOCKED_REQUEST" in str(request),
    )


session = SQLiteSession("notebook-handoff-" + uuid4().hex, ":memory:")
triage = toolkit.Agent(
    "triage",
    runtime=toolkit.runtime(
        provider="openai-runtime",
        model=model_id,
        endpoint=endpoint,
        api_key=os.environ["OPENAI_API_KEY"],
        max_tokens=256,
        max_retries=0,
        timeout_s=40,
    ),
    framework=toolkit.framework(
        "openai-agents",
        agent_kwargs={"handoffs": [transfer], "input_guardrails": [block]},
        run_kwargs={"session": session},
    ),
    instructions="Always transfer control to billing_specialist. Never answer yourself.",
)
try:
    handed_off = await triage.arun(
        "Transfer this billing question to billing_specialist now.",
        timeout_s=90,
    )
    blocked = await triage.arun("BLOCKED_REQUEST", timeout_s=30)
    assert handed_off.ok and "SPECIALIST_OK" in handed_off.text and observed
    assert not blocked.ok
    assert any(
        event.meta.get("kind") == "handoff"
        for node in handed_off.walk()
        for event in node.tool_events
    )
    print(handed_off.human_results())
finally:
    session.close()
    await client.close()